# Data Encoding Examples — Code & Usage 

This notebook demonstrates runnable examples for common categorical encodings and notes on where/how to use them.

In [1]:
# Install requirements (uncomment if needed)
!pip install scikit-learn pandas category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 742.9 kB/s  0:00:13m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [category_encoders]category_encoders]


In [2]:
import pandas as pd

data = pd.DataFrame({
    'Color': ['Red', 'Blue', 'Green', 'Blue', 'Red', 'Green'],
    'Size': ['S', 'M', 'L', 'M', 'S', 'L'],
    'Neighborhood': ['A', 'A', 'B', 'B', 'A', 'C'],
    'Price': [100, 120, 150, 130, 110, 160]
})

data

,Color,Size,Neighborhood,Price
0,Red,S,A,100
1,Blue,M,A,120
2,Green,L,B,150
3,Blue,M,B,130
4,Red,S,A,110
5,Green,L,C,160


## Label Encoding (when: ordinal categories or tree-based models)

In [3]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
data['Color_label'] = le.fit_transform(data['Color'])
data[['Color', 'Color_label']]

,Color,Color_label
0,Red,2
1,Blue,0
2,Green,1
3,Blue,0
4,Red,2
5,Green,1


## One‑Hot Encoding (when: unordered categories with few unique values)

In [5]:
from sklearn.preprocessing import OneHotEncoder

# newer sklearn uses `sparse_output` instead of `sparse`
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_arr = ohe.fit_transform(data[['Color']])
ohe_cols = ohe.get_feature_names_out(['Color'])
ohe_df = pd.DataFrame(ohe_arr, columns=ohe_cols, index=data.index)
pd.concat([data, ohe_df], axis=1)

,Color,Size,Neighborhood,Price,Color_label,Color_Blue,Color_Green,Color_Red
0,Red,S,A,100,2,0.0,0.0,1.0
1,Blue,M,A,120,0,1.0,0.0,0.0
2,Green,L,B,150,1,0.0,1.0,0.0
3,Blue,M,B,130,0,1.0,0.0,0.0
4,Red,S,A,110,2,0.0,0.0,1.0
5,Green,L,C,160,1,0.0,1.0,0.0


## Ordinal Encoding (when: categories have a natural order)

In [6]:
from sklearn.preprocessing import OrdinalEncoder

size_order = [['S', 'M', 'L']]
enc = OrdinalEncoder(categories=size_order)
data['Size_ord'] = enc.fit_transform(data[['Size']])
data[['Size', 'Size_ord']]

,Size,Size_ord
0,S,0.0
1,M,1.0
2,L,2.0
3,M,1.0
4,S,0.0
5,L,2.0


## Binary Encoding (when: many categories, one-hot is too large)

In [7]:
import category_encoders as ce

be = ce.BinaryEncoder(cols=['Color'])
be_df = be.fit_transform(data[['Color']])
pd.concat([data, be_df], axis=1)

,Color,Size,Neighborhood,Price,Color_label,Size_ord,Color_0,Color_1
0,Red,S,A,100,2,0.0,0,1
1,Blue,M,A,120,0,1.0,1,0
2,Green,L,B,150,1,2.0,1,1
3,Blue,M,B,130,0,1.0,1,0
4,Red,S,A,110,2,0.0,0,1
5,Green,L,C,160,1,2.0,1,1


## Practical Tips

- Use simple encodings for low-cardinality features (One‑Hot/Label).
- For high-cardinality features, prefer Binary or Target encodings (with CV and smoothing).
- Always validate encoding choices with cross-validation to avoid leakage.
- Save your encoder objects and reuse them at inference to ensure consistent transforms.